# 👁 level_flow — see the mechanics before trusting a number

This notebook optimizes nothing. It takes **one volatile name** and one **anchor session** and walks the strategy's logic in the order it happens, so that by the end you can judge whether the implemented method is the one the README describes:

| § | question | figures | scope |
|---|---|---|---|
| 1 | which name, which minutes | — | the loaded window (~4 months) |
| 2 | **how are the anchor session's levels built?** — its lookback as coarse candles, the swings that count, the histogram, the resulting table | Fig 2 | the anchor session and the `level_days` sessions before it |
| 3 | **where do those levels sit on the 1-minute tape, and where do the triggers fire?** — the *same* sessions as §2, each with the level table it read, every bounce and break marked, the anchor session highlighted | Fig 3 | the same window as §2 |
| 4 | **where the bounce's level came from, then one bounce and one break up close** — the swings behind the level, then the latest event of each polarity inside §3's window: touch band, signal, fill, stop, target, and what the engine booked | Fig 4a (the level), Fig 4b (bounce), Fig 4c (break) | two events from §3 |
| 5 | **from two anecdotes to every event** — what happened after all the raw triggers of the loaded months | Fig 5 | the loaded window |
| 6 | **is the construction stable?** — the anchor session's table under other candle sizes and bin widths | Fig 6 | the anchor session |

Figures are numbered after the section that draws them (Fig 4b = §4's second figure), and each number is stamped into its figure's title, so it stays attached to a screenshot.

Set `SYMBOL`, `START`/`END`, `SESSION` (the anchor; `None` = the last loaded session) and `PARAMS` (any `config.yaml` knob) at the top of §1. Everything you do not name in `PARAMS` comes from `config.yaml`, never from `signals()`' own defaults. Every event here is **uncapped** — the watchlist and the `max_stop` cap are switched off so you see the raw trigger on this name, not whether it won a seat that day. Levels come only from sessions *before* the one being read; the session's own swings are drawn hollow because it cannot know them yet.

## 1 · Pick a volatile name and load its minutes

In [ ]:
# If this import fails: select the "trading-simple" Jupyter kernel
# (see README "Quick start"), or run  uv sync --extra notebook
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots

from trading_simple import backtest as bt, plots, strategies
from trading_simple.config import load_settings
from trading_simple.data import load_panel, symbols_with_bars
from trading_simple.db import make_engine

settings = load_settings()
engine = make_engine(settings)
strat = strategies.load("level_flow")
mod = strat.signals.__globals__
compose, explain_levels = mod["compose"], mod["explain_levels"]
d = strat.data

SYMBOL = None            # None = the most volatile name below (by average daily range)
START, END = "2026-01-02", d["end"]   # the minutes to load: ~4 months keeps every figure light
SESSION = None           # the session whose level table §2 opens up; None = the last one
PARAMS = {}              # any signals() knob, e.g. {"tf_levels": 30, "bin_pct": 0.003}
# config.yaml under your overrides. Pass THIS to compose()/explain_levels(), never
# nothing: their own defaults differ from the shipped config (tf_levels 30 vs 15),
# so a bare call draws a strategy nobody runs.
PRM = {**strat.params, **PARAMS}
LEVEL_DAYS = PRM["level_days"]

# Volatility from DAILY bars (cheap): average daily range as a share of price over the train
# window, for every name of the universe that has 1-minute bars in [START, END].
universe = settings.universe_symbols(d["universe"])
daily = load_panel(engine, universe, "1d", d["start"], d["end"], exchanges=settings.exchanges)
adr = ((daily.high - daily.low) / daily.close).mean().rename("avg daily range")
dollar = (daily.close * daily.volume).median().rename("median $ volume")
covered = set(symbols_with_bars(engine, universe, "1m", START, END))
rank = (pd.concat([adr, dollar], axis=1).loc[sorted(covered)]
        .sort_values("avg daily range", ascending=False))
display(rank.head(10).style.format({"avg daily range": "{:.1%}", "median $ volume": "${:,.0f}"}))
SYMBOL = SYMBOL or rank.index[0]

panel = load_panel(engine, [SYMBOL], d["bar"], START, END, session=d.get("session", "regular"),
                   exchanges=settings.exchanges, resample=d.get("resample"))
sessions = pd.Series(panel.close.index.tz_convert(panel.tz).date, index=panel.close.index)
print(f"{SYMBOL}: {len(panel.close):,} {panel.bar} bars over {sessions.nunique()} sessions "
      f"({sessions.iloc[0]} → {sessions.iloc[-1]}), average daily range "
      f"{adr[SYMBOL]:.1%} (universe median {adr.median():.1%})")

## 2 · How one session's levels are built (Fig 2)

**Fig 2**, left: the `level_days` sessions before `SESSION` plus the session itself, as `tf_levels`-minute candles (15 in `config.yaml`: only the *level construction* looks at coarser candles — on 1-minute bars almost every bar is a local extreme of its neighbours and the histogram fills with noise instead of turning points; triggers, fills and stops all run on the 1-minute bars. Set `PARAMS = {"tf_levels": 5}` or `30` to see the difference, and §6 compares three sizes side by side). **▲ / ▼ = swing high / low** — a candle whose high (low) is the highest (lowest) of the `pivot_k` candles on each side. A swing is only *confirmed* `pivot_k` candles later, so a swing near a session's close is known to the next session, and today's own swings (drawn hollow) are not known at all. Right: the **histogram** — how many confirmed swings fell into each `bin_pct`-wide log-price bin over the lookback; a bin at or above `min_touches` (the vertical line) is a **level**, drawn as a horizontal line across the session at the mean price of its swings. Below: the table the session actually reads.

What to look for: levels should sit where the eye sees repeated turning points, not on every wick; a level with many touches should be one the price *visibly* respected before; and nothing drawn should use a swing to its right.

In [ ]:
x = explain_levels(panel, SYMBOL, SESSION, **PRM)
day, cnd, piv, lv, hist = x["session"], x["candles"], x["pivots"], x["levels"], x["histogram"]
p = x["params"]
this = cnd[cnd["session"] == cnd["session"].max()]
t0, t1 = this["ts"].iloc[0], this["ts"].iloc[-1] + pd.Timedelta(minutes=p["tf_levels"])

fig = make_subplots(rows=1, cols=2, shared_yaxes=True, column_widths=[0.78, 0.22],
                    horizontal_spacing=0.02,
                    subplot_titles=(f"{SYMBOL} — {p['tf_levels']}-minute candles, "
                                    f"{LEVEL_DAYS} sessions before {day} + the session",
                                    "swings per price bin"))
fig.add_trace(go.Candlestick(x=cnd["ts"], open=cnd["open"], high=cnd["high"], low=cnd["low"],
                             close=cnd["close"], name="candles", showlegend=False,
                             increasing_line_color=plots.SERIES[1],
                             decreasing_line_color=plots.SERIES[5]), row=1, col=1)
for side, sym, color, dy in (("high", "triangle-down", plots.SERIES[5], 1),
                             ("low", "triangle-up", plots.SERIES[3], -1)):
    for counted, fill in ((True, color), (False, "white")):
        pts = piv[(piv["side"] == side) & (piv["counted"] == counted)]
        if len(pts):
            fig.add_trace(go.Scatter(
                x=pts["ts"], y=pts["price"] * (1 + dy * 0.002), mode="markers",
                name=f"swing {side}" + (" (confirmed)" if counted else " (not yet)"),
                marker={"symbol": sym, "size": 10, "color": fill,
                        "line": {"color": color, "width": 1.5}},
                customdata=pts[["session", "known_session"]],
                hovertemplate=(f"swing {side} %{{y:.2f}}<br>session %{{customdata[0]}}, "
                               "confirmed in %{customdata[1]}<extra></extra>")), row=1, col=1)
# Shapes go on AFTER the traces of their subplot, or plotly drops them silently. Each level
# is dotted across the lookback it was built from (so the swings it came from line up with
# it) and solid across the session that reads it.
for _, r in lv.iterrows():
    width = 1.0 + 0.5 * min(r["touches"], 6)
    fig.add_shape(type="line", x0=cnd["ts"].iloc[0], x1=t0, y0=r["level"], y1=r["level"],
                  row=1, col=1, line={"color": plots.SERIES[0], "width": width, "dash": "dot"},
                  opacity=0.55, layer="below")
    fig.add_shape(type="line", x0=t0, x1=t1, y0=r["level"], y1=r["level"], row=1, col=1,
                  line={"color": plots.SERIES[0], "width": width + 0.5}, layer="above")
    fig.add_annotation(x=t1, y=r["level"], text=f"{r['touches']:.0f}", showarrow=False,
                       xanchor="left", font={"size": 10, "color": plots.SERIES[0]}, row=1, col=1)
fig.add_trace(go.Bar(y=(hist["bin_low"] + hist["bin_high"]) / 2, x=hist["touches"],
                     orientation="h", width=(hist["bin_high"] - hist["bin_low"]).to_numpy(),
                     marker={"color": np.where(hist["touches"] >= max(p["min_touches"], 1),
                                               plots.SERIES[0], "#c9c7bf")},
                     name="swings in bin", showlegend=False,
                     hovertemplate="%{y:.2f}: %{x} swings<extra></extra>"), row=1, col=2)
fig.add_vline(x=max(p["min_touches"], 1) - 0.5, line={"color": "#898781", "dash": "dot"},
              row=1, col=2)
plots._base(fig, title=f"<b>Fig 2</b> · {SYMBOL} · level table read on {day}: {len(lv)} levels "
                       f"(bin {p['bin_pct']:.2%}, ≥ {p['min_touches']} swings)", height=600)
# The shared layout puts a horizontal legend right above the panels, where it collides with
# the subplot titles; park it beside the panels and give the title its own band (plot_symbol
# does the same).
fig.update_layout(legend={"orientation": "v", "yanchor": "top", "y": 1.0, "xanchor": "left",
                          "x": 1.005}, margin={"r": 230, "t": 90})
fig.update_annotations(yshift=6)
fig.update_xaxes(rangeslider_visible=False, type="date", row=1, col=1,
                 rangebreaks=plots._intraday_breaks(panel.close.index[
                     (panel.close.index >= cnd["ts"].iloc[0]) & (panel.close.index <= t1)]))
fig.update_xaxes(title_text="swings", row=1, col=2)
fig.update_yaxes(title_text="price ($)", row=1, col=1)
fig.show()
print(f"levels read on {day} (session ATR {x['atr'].loc[str(day)].iloc[0]:.2f}):")
display(lv.style.format({"level": "{:.2f}", "bin_low": "{:.2f}", "bin_high": "{:.2f}",
                         "touches": "{:.0f}"}))

## 3 · The same sessions on the 1-minute tape: the levels each one read, and where the triggers fire (Fig 3)

**Fig 3**'s window is exactly Fig 2's: the anchor session (shaded) and the `level_days` sessions before it. For every session, the **level table it read** is drawn as horizontal segments across that session — thicker for more swings behind the level — so Fig 2's levels can be followed onto the tape and compared with the price action of the day. Markers are the **raw triggers**, watchlist off and the `max_stop` cap lifted (the printout says how many the shipped cap would keep), both polarities computed side by side on the identical levels: ▲ bounce-long (blue) / ▼ bounce-short (orange) fire on a reclaim after the price entered a level's touch band; ▲ break-long (green) / ▼ break-short (red) fire on a close through the level in force at the previous bar, with the session's first close on the inside. One trigger per session and side. Hover a marker for the level it rests on and that level's swing count. Times are New York.

What to look for: the anchor session's segments are §2's table, line for line; a bounce marker should sit within a touch band of a level (just above a support, just below a resistance) after the price reached that band; a break marker just beyond a level the price opened inside of; no marker in the first `scan_minute` minutes or the last `skip_close`; and levels should be *where turning points repeat*, not on every wick.

In [ ]:
# The watchlist and the max_stop cap off; every other knob is the shipped config's.
RAW = {**strat.params, "max_candidates": len(panel.symbols), "vol_mult": 0.0,
       "min_price": 0.0, "min_dollar_volume": 0.0, "max_stop": 1.0, **PARAMS}
parts = {pol: compose(panel, **{**RAW, "polarity": pol}) for pol in ("bounce", "break")}
# The shipped stop cap, applied to this name: a 0.3-ATR stop on a 10%-range name is already
# ~3% of price, so `max_stop` alone can silence a volatile name — worth knowing before
# reading the markers as "the strategy would have traded here".
capped = {pol: compose(panel, **{**RAW, "polarity": pol,
                                 "max_stop": strat.params["max_stop"]})
          for pol in ("bounce", "break")}


def total(c):
    return int(c["long_raw"].to_numpy().sum() + c["short_raw"].to_numpy().sum())


for pol in ("bounce", "break"):
    print(f"{pol}: {total(parts[pol])} raw triggers; {total(capped[pol])} survive the shipped "
          f"max_stop {strat.params['max_stop']:.1%} of price")

close = panel.close[SYMBOL]
local = close.index.tz_convert(panel.tz).tz_localize(None)     # New York wall clock
day_of = pd.Series(local.date, index=close.index)
EVENTS = {("bounce", "long"): ("triangle-up", plots.SERIES[0]),
          ("bounce", "short"): ("triangle-down", plots.SERIES[7]),
          ("break", "long"): ("triangle-up", plots.SERIES[3]),
          ("break", "short"): ("triangle-down", plots.SERIES[5])}


def trigger_bars(pol, side):
    """Signal bars (before the next-bar shift) of one polarity and side, on this name."""
    return parts[pol][f"{side}_raw"][SYMBOL]


# The window of §2: the anchor session and the level_days sessions before it.
all_days = sorted(day_of.unique())
anchor = all_days.index(day)
shown = all_days[max(anchor - LEVEL_DAYS, 0):anchor + 1]
win = day_of.isin(shown).to_numpy()
tape, t_local = close[win], local[win]
tables = {s: (lv if s == day else explain_levels(panel, SYMBOL, s, **PRM)["levels"])
          for s in shown}

fig = make_subplots(rows=2, cols=1, shared_xaxes=True, row_heights=[0.78, 0.22],
                    vertical_spacing=0.04, subplot_titles=("", "relative volume at each bar"))
# Each session's level table as segments across that session; three traces by swing count.
for lo_n, hi_n, width, label in ((2, 2, 1.0, "level (2 swings)"), (3, 4, 2.0, "level (3-4 swings)"),
                                 (5, 99, 3.0, "level (5+ swings)")):
    xs, ys, text = [], [], []
    for s in shown:
        bars = t_local[day_of[win].to_numpy() == s]
        for _, r in tables[s].iterrows():
            if lo_n <= r["touches"] <= hi_n:
                xs += [bars[0], bars[-1], None]
                ys += [r["level"], r["level"], None]
                text += [f"{r['touches']:.0f} swings"] * 3
    if xs:
        fig.add_trace(go.Scatter(x=xs, y=ys, mode="lines", name=label, text=text,
                                 line={"color": plots.SERIES[0], "width": width}, opacity=0.55,
                                 hovertemplate="level %{y:.2f} · %{text}<extra></extra>"),
                      row=1, col=1)
fig.add_trace(go.Scatter(x=t_local, y=tape, mode="lines", name="close",
                         line={"color": "#0b0b0b", "width": 1},
                         hovertemplate="%{x|%b %d %H:%M}<br>%{y:.2f}<extra>close</extra>"),
              row=1, col=1)
for (pol, side), (sym, color) in EVENTS.items():
    mask = trigger_bars(pol, side).to_numpy() & win
    if mask.any():
        n_lvl = (parts[pol]["sup_touches"] if side == "long" else parts[pol]["res_touches"])[SYMBOL]
        fig.add_trace(go.Scatter(
            x=local[mask], y=close[mask], mode="markers", name=f"{pol} {side}",
            marker={"symbol": sym, "size": 12, "color": color,
                    "line": {"color": "white", "width": 1}},
            customdata=np.c_[parts[pol]["level_" + side][SYMBOL][mask], n_lvl[mask]],
            hovertemplate=(f"{pol} {side} %{{x|%b %d %H:%M}} @ %{{y:.2f}}<br>level "
                           "%{customdata[0]:.2f} (%{customdata[1]:.0f} swings)<extra></extra>")),
            row=1, col=1)
rvol = parts["bounce"]["rvol"][SYMBOL]
fig.add_trace(go.Scatter(x=t_local, y=rvol[win], mode="lines", name="rvol", showlegend=False,
                         line={"color": plots.SERIES[4], "width": 1},
                         hovertemplate="%{x|%b %d %H:%M}<br>%{y:.2f}x<extra>rvol</extra>"),
              row=2, col=1)
counts = {f"{pol}-{side}": int((trigger_bars(pol, side).to_numpy() & win).sum())
          for (pol, side) in EVENTS}
fired = ", ".join(f"{k} {v}" for k, v in counts.items())
plots._base(fig, title=f"<b>Fig 3</b> · {SYMBOL} — Fig 2's window on the tape: {len(shown)} "
                       f"sessions ending with the anchor {day}, each session's level table, "
                       f"every raw trigger ({fired})", height=760)
# Shapes go on after the traces they belong with: faint separators at each session's open, and
# the anchor session shaded so §2's table can be found at a glance.
for s in shown[1:]:
    fig.add_vline(x=t_local[day_of[win].to_numpy() == s][0], line={"color": "#e1e0d9", "width": 1},
                  row=1, col=1)
anchor_bars = t_local[day_of[win].to_numpy() == day]
fig.add_vrect(x0=anchor_bars[0], x1=anchor_bars[-1], fillcolor=plots.SERIES[2], opacity=0.07,
              line_width=0, layer="below", row=1, col=1,
              annotation_text=f"anchor session {day}", annotation_position="top left",
              annotation_font_size=11, annotation_font_color="#898781")
fig.update_layout(legend={"orientation": "v", "yanchor": "top", "y": 1.0, "xanchor": "left",
                          "x": 1.005}, margin={"r": 220, "t": 70})
fig.update_xaxes(type="date", rangebreaks=plots._intraday_breaks(t_local),
                 title_text="New York time", row=2, col=1)
fig.update_yaxes(title_text="price ($)", row=1, col=1)
fig.show()

# One row per session: what it opened at, what it read, what fired.
first_bar = lambda s: s.groupby(day_of.to_numpy()).nth(0).to_numpy()  # noqa: E731
sup, res = parts["bounce"]["support"][SYMBOL], parts["bounce"]["resistance"][SYMBOL]
per_session = pd.DataFrame({
    "open": first_bar(panel.open_[SYMBOL]),
    "close": close.groupby(day_of.to_numpy()).last().to_numpy(),
    "session ATR": first_bar(parts["bounce"]["atr"][SYMBOL]),
    "levels in table": [len(tables[s]) if s in tables else np.nan for s in all_days],
    "nearest below at open": first_bar(sup), "nearest above at open": first_bar(res),
    **{f"{pol} {'▲' if side == 'long' else '▼'}":
       trigger_bars(pol, side).groupby(day_of.to_numpy()).sum().to_numpy()
       for (pol, side) in EVENTS},
}, index=all_days)
display(per_session.loc[shown].style.format({
    "open": "{:.2f}", "close": "{:.2f}", "session ATR": "{:.2f}", "levels in table": "{:.0f}",
    "nearest below at open": "{:.2f}", "nearest above at open": "{:.2f}"}, na_rep="—"))

## 4 · The level under the trade, then the two events up close (Figs 4a, 4b, 4c)

**Fig 4a** answers *where did this level come from?* for the bounce §4 inspects, in the three steps the code takes. **1** — every confirmed swing of the `level_days` sessions behind the level, grey, over the coarse candles' closes; the swings **inside the level's own bin** are blue and dated, and the shaded strip is that bin. **2** — the pivot rule itself, on the latest of those swings: its `tf_levels`-minute candle in blue among the `pivot_k` candles each side, whose extremes never trade through it, and the note that it is only *confirmed* `pivot_k` candles later, so an earlier session could not have used it. **3** — the same swings counted into `bin_pct`-wide log-price bins across the whole lookback: the bin that cleared `min_touches` is blue, and the level is the mean price of the swings in it. The three panels share one price axis, so a bin in panel 3 lines up with the swings in panel 1. The trade that used the level is Fig 4b.

**Figs 4b and 4c** are the **latest bounce (4b) and the latest break (4c) inside Fig 3's window** (so they sit in or near the anchor session; if a polarity never fired in the window, the latest one before it is shown and said so). Each with the level it rests on, the **touch band the trigger tests** (bounce-long: the `touch_tol` session-ATRs *above* a support, the low has to get in there, or below it; bounce-short: the band *below* a resistance; break: the `break_tol` margin the close must clear), the touch bar (◆), the signal bar, the fill (next bar's open), the stop (`sl_atr` beyond the level, on the far side) and the target (`tp_ratio` × the stop distance). The window runs from before the signal to the bar the engine would have booked the exit on — stop, target, the time stop or the session close — and the exit marker sits where the engine would have got out: a stop or target at its level (at the bar's open when it gapped through), a time stop or an end-of-day flat at that bar's **open**, never at its close. Set `EVENT_BOUNCE` / `EVENT_BREAK` to any New York timestamp from Fig 3's hover to inspect a specific one.

What to look for: the swings behind a level should be spread over several sessions rather than all come from one; a bounce marker sitting *below* a support's band, or a break marker on the wrong side of its level, is a bug. A bounce fired on a bar whose extreme only *grazed* the band's far edge is not a bug but a loose `touch_tol` — the study searches it.


In [ ]:
EVENT_BOUNCE = None      # e.g. "2026-04-22 10:42" (New York); None = latest bounce in §3's window
EVENT_BREAK = None       # e.g. "2026-04-24 10:05"; None = latest break in §3's window
BEFORE, AFTER = 60, 15   # bars drawn before the signal, and after the exit
prm = RAW            # RAW already carries config.yaml
mpb = mod["_minutes_per_bar"](panel.bar)
horizon_b = int(np.ceil(prm["horizon"] / mpb))
touch_b = max(int(np.ceil(prm["touch_minutes"] / mpb)), 1)
idx = close.index
pos = pd.Series(np.arange(len(idx)), index=idx)
end_bar = bt.session_end(idx, panel.tz).to_numpy()
opens, highs, lows = (panel.open_[SYMBOL].to_numpy(), panel.high[SYMBOL].to_numpy(),
                      panel.low[SYMBOL].to_numpy())


def pick_event(pol, when):
    """The (bar position, side) of a trigger: the given timestamp, else the latest in §3."""
    if when is not None:
        i = int(pos[pd.Timestamp(when, tz=panel.tz).tz_convert("UTC")])
        for side in ("long", "short"):
            if trigger_bars(pol, side).iloc[i]:
                return i, side
        raise KeyError(f"no {pol} trigger at {when}")
    for scope, mask_extra in (("§3's window", win), ("the loaded months", np.ones(len(idx), bool))):
        latest = [(int(np.flatnonzero(trigger_bars(pol, side).to_numpy() & mask_extra)[-1]), side)
                  for side in ("long", "short")
                  if (trigger_bars(pol, side).to_numpy() & mask_extra).any()]
        if latest:
            if scope != "§3's window":
                print(f"no {pol} trigger inside §3's window — showing the latest one before it")
            return max(latest)
    raise KeyError(f"no {pol} trigger in the loaded months")


def resolve(i, side, stop_px, target_px):
    """Where the engine books the exit of a fill at bar i+1: ``(bar, cause, price)``.

    A stop or target fills at its own level — at the bar's OPEN when the bar gapped
    through it, which is what `bt.run` asks vectorbt for. The time stop lands `horizon`
    bars after the FILL (`bt.exit_after` shifts the entry frame) and the end-of-day flat
    on the session's last bar; both fill at that bar's open, never at its close, which is
    not a price the engine can trade.
    """
    last = min(i + 1 + horizon_b, len(idx) - 1)
    for j in range(i + 1, last + 1):
        hit_stop = lows[j] <= stop_px if side == "long" else highs[j] >= stop_px
        hit_tgt = target_px is not None and (highs[j] >= target_px if side == "long"
                                             else lows[j] <= target_px)
        if hit_stop:
            gapped = opens[j] <= stop_px if side == "long" else opens[j] >= stop_px
            return j, "stop", opens[j] if gapped else stop_px
        if hit_tgt:
            gapped = opens[j] >= target_px if side == "long" else opens[j] <= target_px
            return j, "target", opens[j] if gapped else target_px
        if end_bar[j] or j == last:
            return j, "session close" if end_bar[j] else "time stop", opens[j]
    return last, "time stop", opens[last]        # only if the data ends inside the horizon


def zoom(pol, when, fig_no):
    """Draw one trigger up close as figure `fig_no`: level, band, fill, stop, target, exit."""
    i, side = pick_event(pol, when)
    level = parts[pol]["level_" + side][SYMBOL].iloc[i]
    atr = parts[pol]["atr"][SYMBOL].iloc[i]
    fill = opens[i + 1]
    dist = parts[pol]["signals"]["sl_stop"][SYMBOL].iloc[i + 1] * fill
    stop_px = fill - dist if side == "long" else fill + dist
    reach = prm["tp_ratio"] * dist
    target_px = (fill + reach if side == "long" else fill - reach) if prm["tp_ratio"] else None
    j, cause, exit_px = resolve(i, side, stop_px, target_px)
    w = slice(max(i - BEFORE, 0), min(j + AFTER, len(idx) - 1) + 1)
    x = local[w]
    fig = go.Figure(go.Candlestick(
        x=x, open=opens[w], high=highs[w], low=lows[w], close=close.iloc[w], name="1-minute",
        increasing_line_color=plots.SERIES[1], decreasing_line_color=plots.SERIES[5]))
    tol = prm["touch_tol"] * atr
    if pol == "bounce":            # the band the trigger tests: the near side of the level
        band = (level, level + tol) if side == "long" else (level - tol, level)
        fig.add_hrect(y0=band[0], y1=band[1], fillcolor=plots.SERIES[0], opacity=0.12,
                      line_width=0, layer="below")
        ext = lows if side == "long" else highs
        hit = [k for k in range(max(i - touch_b + 1, 0), i + 1)
               if (ext[k] <= level + tol if side == "long" else ext[k] >= level - tol)]
        if hit:
            k = hit[0]
            fig.add_trace(go.Scatter(x=[local[k]], y=[ext[k]], mode="markers", name="touch",
                                     marker={"symbol": "diamond", "size": 11,
                                             "color": plots.SERIES[0],
                                             "line": {"color": "white", "width": 1}}))
    else:
        margin = prm["break_tol"] * atr
        y0, y1 = (level, level + margin) if side == "long" else (level - margin, level)
        fig.add_hrect(y0=y0, y1=y1, fillcolor=plots.SERIES[2], opacity=0.15, line_width=0,
                      layer="below")
    for y, name, color, dash in ((level, f"level {level:.2f}", plots.SERIES[0], "solid"),
                                 (stop_px, f"stop {stop_px:.2f}", plots.SERIES[5], "dash"),
                                 (target_px, f"target {target_px:.2f}" if target_px else "",
                                  plots.SERIES[3], "dash")):
        if y is not None:
            fig.add_hline(y=y, line={"color": color, "width": 1.5, "dash": dash},
                          annotation_text=name, annotation_position="top left")
    sym, color = EVENTS[(pol, side)]
    fig.add_trace(go.Scatter(x=[local[i]], y=[close.iloc[i]], mode="markers",
                             name=f"{pol} {side} signal",
                             marker={"symbol": sym, "size": 14, "color": color,
                                     "line": {"color": "white", "width": 1}}))
    fig.add_trace(go.Scatter(x=[local[i + 1]], y=[fill], mode="markers", name=f"fill @ {fill:.2f}",
                             marker={"symbol": "circle", "size": 9, "color": "#0b0b0b"}))
    fig.add_trace(go.Scatter(x=[local[j]], y=[exit_px], mode="markers",
                             name=f"exit: {cause} @ {exit_px:.2f}",
                             marker={"symbol": "x", "size": 12, "color": "#898781",
                                     "line": {"color": "#898781", "width": 2}}))
    ret = (exit_px / fill - 1) * (1 if side == "long" else -1)
    plots._base(fig, title=f"<b>Fig {fig_no}</b> · {SYMBOL} · {pol} {side} signal at "
                           f"{local[i]:%Y-%m-%d %H:%M} New York — {cause} {j - i - 1} bars after "
                           f"the fill ({ret:+.2%})", height=520)
    fig.update_xaxes(rangeslider_visible=False, type="date", title_text="New York time",
                     rangebreaks=plots._intraday_breaks(x))
    fig.update_layout(legend={"orientation": "v", "yanchor": "top", "y": 1.0, "xanchor": "left",
                              "x": 1.005}, margin={"r": 170})
    fig.show()

In [ ]:
# Fig 4a — how the level under the §4 bounce was built, in the three steps the code takes.
# Same event as Fig 4b, so the two figures always describe one trade.
i_b, side_b = pick_event("bounce", EVENT_BOUNCE)
ev_day = local[i_b].date()
ev_level = parts["bounce"]["level_" + side_b][SYMBOL].iloc[i_b]
ev_atr = parts["bounce"]["atr"][SYMBOL].iloc[i_b]
xe = explain_levels(panel, SYMBOL, ev_day, **PRM)
lv = xe["levels"].loc[(xe["levels"]["level"] - ev_level).abs().idxmin()]
lo_b, hi_b = lv["bin_low"], lv["bin_high"]
k = PRM["pivot_k"]

cnd_e, piv_e = xe["candles"], xe["pivots"]
cnd_x = pd.DatetimeIndex(cnd_e["ts"]).tz_convert(panel.tz).tz_localize(None)
piv_x = pd.DatetimeIndex(piv_e["ts"]).tz_convert(panel.tz).tz_localize(None)
before = (cnd_e["session"] < cnd_e["session"].max()).to_numpy()    # the lookback, not today
in_bin = (piv_e["counted"] & (piv_e["price"] >= lo_b) & (piv_e["price"] < hi_b)).to_numpy()
other = piv_e["counted"].to_numpy() & ~in_bin
n_in = int(in_bin.sum())
# Step 2 opens up the LATEST of the swings the level is made of.
def _one_session(j):
    """True when the pivot window of swing `j` does not cross a session boundary."""
    a = int(np.flatnonzero(cnd_x == piv_x[j])[0])
    return cnd_e.iloc[max(a - k, 0):a + k + 1]["session"].nunique() == 1


marks = np.flatnonzero(in_bin)
sw = next((int(j) for j in marks[::-1] if _one_session(int(j))), int(marks[-1]))
sw_x, sw_px, sw_side = piv_x[sw], float(piv_e["price"].iloc[sw]), piv_e["side"].iloc[sw]
at = int(np.flatnonzero(cnd_x == sw_x)[0])
lo_i, hi_i = max(at - k - 1, 0), min(at + k + 1, len(cnd_e) - 1)
z, z_x = cnd_e.iloc[lo_i:hi_i + 1], cnd_x[lo_i:hi_i + 1]
z_lab = [f"{t:%H:%M}" for t in z_x]                 # a category axis: no session gaps
z_ext = z["low"] if sw_side == "low" else z["high"]

fig = make_subplots(rows=1, cols=3, column_widths=[0.44, 0.31, 0.25], horizontal_spacing=0.06,
                    subplot_titles=(f"1 · the swings of {LEVEL_DAYS} sessions",
                                    "2 · what makes a swing", "3 · swings per bin"))
# --- 1 · the raw material: the confirmed swings of the lookback, and the two in this bin ---
fig.add_trace(go.Scatter(x=cnd_x[before], y=cnd_e["close"][before], mode="lines",
                         name="candle closes", showlegend=False, opacity=0.5,
                         line={"color": "#c9c7bf", "width": 1}, hoverinfo="skip"), row=1, col=1)
fig.add_trace(go.Scatter(
    x=piv_x[other], y=piv_e["price"][other], mode="markers", name="confirmed swings",
    marker={"symbol": np.where(piv_e["side"][other] == "high", "triangle-down", "triangle-up"),
            "size": 8, "color": "#898781"},
    hovertemplate="swing %{y:.2f}<br>%{x|%b %d %H:%M}<extra></extra>"), row=1, col=1)
fig.add_trace(go.Scatter(
    x=piv_x[in_bin], y=piv_e["price"][in_bin], mode="markers+text",
    name=f"the {n_in} swings inside this bin",
    text=[f" {t:%b %d} " for t in piv_x[in_bin]], textposition="top center",
    textfont={"size": 11, "color": plots.SERIES[0]},
    marker={"symbol": np.where(piv_e["side"][in_bin] == "high", "triangle-down", "triangle-up"),
            "size": 15, "color": plots.SERIES[0], "line": {"color": "white", "width": 1}},
    customdata=piv_e["side"][in_bin],
    hovertemplate="swing %{customdata} %{y:.2f}<br>%{x|%b %d %H:%M}<extra></extra>"),
    row=1, col=1)
# --- 2 · the pivot rule, on the latest of them ---------------------------------------------
fig.add_trace(go.Candlestick(x=z_lab, open=z["open"], high=z["high"], low=z["low"],
                             close=z["close"], name="candles", showlegend=False,
                             increasing_line_color="#8f8d85", decreasing_line_color="#8f8d85",
                             line={"width": 1.4}), row=1, col=2)
one = z.iloc[[at - lo_i]]
fig.add_trace(go.Candlestick(x=[z_lab[at - lo_i]], open=one["open"], high=one["high"],
                             low=one["low"], close=one["close"], showlegend=False,
                             name="the swing candle", line={"width": 1.8},
                             increasing_line_color=plots.SERIES[0],
                             decreasing_line_color=plots.SERIES[0]), row=1, col=2)
fig.add_trace(go.Scatter(
    x=[f"{sw_x:%H:%M}"], y=[sw_px], mode="markers", showlegend=False,
    marker={"symbol": "triangle-up" if sw_side == "low" else "triangle-down", "size": 16,
            "color": plots.SERIES[0], "line": {"color": "white", "width": 1}},
    hovertemplate=f"the swing {sw_side} %{{y:.2f}}<extra></extra>"), row=1, col=2)
fig.add_trace(go.Scatter(x=z_lab, y=z_ext, mode="markers", showlegend=False,
                         marker={"symbol": "line-ew", "size": 9,
                                 "line": {"color": "#898781", "width": 1.5}},
                         hovertemplate=f"{sw_side} %{{y:.2f}}<extra></extra>"), row=1, col=2)
# --- 3 · the histogram every bin of the lookback is counted into --------------------------
hist_e = xe["histogram"]
fig.add_trace(go.Bar(
    y=(hist_e["bin_low"] + hist_e["bin_high"]) / 2, x=hist_e["touches"], orientation="h",
    width=(hist_e["bin_high"] - hist_e["bin_low"]).to_numpy(), showlegend=False, name="swings",
    marker={"color": np.where(np.isclose(hist_e["bin_low"], lo_b), plots.SERIES[0], "#d5d3cc")},
    hovertemplate="%{y:.2f}: %{x} swings<extra></extra>"), row=1, col=3)
plots._base(fig, title=f"<b>Fig 4a</b> · {SYMBOL} · how the {lv['level']:.2f} level under the "
                       f"{local[i_b]:%b %d %H:%M} bounce {side_b} was built", height=560)
fig.update_layout(legend={"orientation": "v", "yanchor": "top", "y": 1.0, "xanchor": "left",
                          "x": 1.005}, margin={"r": 210, "t": 90})
fig.update_annotations(yshift=6, font_size=12)   # subplot titles, before my own annotations
# Shapes and their labels go on AFTER the traces of their subplot, or plotly drops them.
for c in (1, 2, 3):
    fig.add_hrect(y0=lo_b, y1=hi_b, row=1, col=c, fillcolor=plots.SERIES[0], opacity=0.12,
                  line_width=0, layer="below")
    fig.add_hline(y=lv["level"], row=1, col=c, line={"color": plots.SERIES[0], "width": 1.5})
fig.add_vline(x=max(PRM["min_touches"], 1) - 0.5, row=1, col=3,
              line={"color": "#898781", "dash": "dot"})
fig.add_hline(y=sw_px, row=1, col=2, line={"color": "#898781", "width": 1, "dash": "dot"})
fig.add_annotation(x=0.02, y=sw_px, xref="x domain", row=1, col=2, xanchor="left",
                   yanchor="bottom", yshift=4, showarrow=False, bgcolor="white",
                   font={"size": 11, "color": plots.SERIES[0]},
                   text=f"{'lowest low' if sw_side == 'low' else 'highest high'} of the {k} "
                        f"candles each side, {sw_x:%b %d %H:%M}")
fig.add_annotation(x=0.5, y=1.0, xref="x domain", yref="y domain", row=1, col=2,
                   xanchor="center", yanchor="top", showarrow=False, bgcolor="white",
                   font={"size": 10, "color": "#898781"},
                   text=f"confirmed {k} candles later, so a level only from the "
                        f"{all_days[int(piv_e['known_session'].iloc[sw])]} session on")
fig.add_annotation(x=lv["touches"], y=lv["level"], row=1, col=3, xanchor="left", yanchor="middle",
                   xshift=6, showarrow=False, bgcolor="white",
                   font={"size": 11, "color": plots.SERIES[0]},
                   text=f"{n_in} ≥ min_touches {PRM['min_touches']} → "
                        f"level {lv['level']:.2f}")
fig.update_xaxes(type="date", row=1, col=1,
                 rangebreaks=plots._intraday_breaks(local[day_of.isin(
                     all_days[max(all_days.index(ev_day) - LEVEL_DAYS, 0):
                              all_days.index(ev_day)]).to_numpy()]))
span = f"{z_x[0]:%b %d}" + ("" if z_x[0].date() == z_x[-1].date() else f"–{z_x[-1]:%b %d}")
fig.update_xaxes(type="category", rangeslider_visible=False, row=1, col=2,
                 title_text=f"{span}, New York")
fig.update_yaxes(range=[min(float(z["low"].min()), lo_b) * 0.995,
                        float(z["high"].max()) * 1.004], row=1, col=2)
fig.update_xaxes(title_text="swings in bin", row=1, col=3)
fig.update_yaxes(title_text="price ($)", row=1, col=1)
fig.update_yaxes(matches="y", showticklabels=False, row=1, col=3)
fig.show()

display(pd.DataFrame({
    "swing": [f"{t:%a %b %d, %H:%M}" for t in piv_x[in_bin]],
    "side": piv_e["side"][in_bin].to_numpy(),
    "price": piv_e["price"][in_bin].to_numpy(),
    "confirmed in session": [str(all_days[int(s)]) for s in piv_e["known_session"][in_bin]],
}).style.format({"price": "{:.2f}"}).hide(axis="index"))
tol_e = prm["touch_tol"] * ev_atr
band = ((lv["level"], lv["level"] + tol_e) if side_b == "long"
        else (lv["level"] - tol_e, lv["level"]))
gap = abs(close.iloc[i_b] - lv["level"])
print(f"level {lv['level']:.2f}: the mean of {n_in} swings inside {lo_b:.2f}-{hi_b:.2f} "
      f"(bin {PRM['bin_pct']:.2%} wide, a level needs {PRM['min_touches']})")
print(f"session ATR {ev_atr:.2f} -> touch band {prm['touch_tol']} x ATR = ${tol_e:.2f} "
      f"({tol_e / lv['level']:.2%} of price), {band[0]:.2f} to {band[1]:.2f} (Fig 4b)")
print(f"trigger close {close.iloc[i_b]:.2f} is ${gap:.2f} ({gap / ev_atr:.2f} ATR) from the "
      f"level; its stop {prm['sl_atr']} ATR beyond = "
      f"{(gap + prm['sl_atr'] * ev_atr) / close.iloc[i_b]:.2%} of price, against the shipped "
      f"max_stop {strat.params['max_stop']:.1%}")


In [ ]:
zoom("bounce", EVENT_BOUNCE, "4b")
zoom("break", EVENT_BREAK, "4c")


## 5 · From two events to every event on this name (Fig 5)

Figs 4b and 4c are two anecdotes. This section follows **every raw trigger of the loaded months** forward from its fill: the mean signed return path (**Fig 5**; long as-is, shorts flipped) over the next three hours, and a table of outcomes — the share that reached the target before the stop, hit the stop first, or ran into the session close / time stop — using the strategy's own stop and target distances. One name and a few months is a small sample (read `n`), and there is no market removed here; the optimize notebook's §5b does that across the whole universe. This is the mechanism check: **do bounces bounce, do breaks run, and by how much relative to the stop?**

In [ ]:
HORIZON = 180
paths, rows = {}, {}
for (pol, side), _ in EVENTS.items():
    bars = np.flatnonzero(trigger_bars(pol, side).to_numpy())
    bars = bars[bars + HORIZON + 1 < len(idx)]
    if not len(bars):
        continue
    sgn = 1 if side == "long" else -1
    fills = opens[bars + 1]
    fwd = np.stack([(opens[bars + 1 + h] / fills - 1) * sgn for h in range(HORIZON + 1)], axis=1)
    paths[f"{pol} {side}"] = fwd.mean(axis=0)
    stop_frac = parts[pol]["signals"]["sl_stop"][SYMBOL].to_numpy()[bars + 1]
    outcome = []
    for b, f, sf in zip(bars, fills, stop_frac, strict=True):
        stop_px = f * (1 - sf) if side == "long" else f * (1 + sf)
        reach = prm["tp_ratio"] * sf
        tgt = (f * (1 + reach) if side == "long" else f * (1 - reach)) if prm["tp_ratio"] else None
        outcome.append(resolve(int(b), side, stop_px, tgt)[1])
    outcome = pd.Series(outcome)
    rows[f"{pol} {side}"] = {
        "n": len(bars), "mean 30 bars (bps)": 1e4 * fwd[:, 30].mean(),
        "mean 60 bars (bps)": 1e4 * fwd[:, 60].mean(),
        "mean 120 bars (bps)": 1e4 * fwd[:, 120].mean(),
        "median 60 bars (bps)": 1e4 * np.median(fwd[:, 60]),
        "target first": (outcome == "target").mean(), "stop first": (outcome == "stop").mean(),
        "time / close": outcome.isin(["time stop", "session close"]).mean(),
        "mean stop (%)": 100 * stop_frac.mean()}
FMT = {"n": "{:.0f}", "mean 30 bars (bps)": "{:+.1f}", "mean 60 bars (bps)": "{:+.1f}",
       "mean 120 bars (bps)": "{:+.1f}", "median 60 bars (bps)": "{:+.1f}",
       "target first": "{:.0%}", "stop first": "{:.0%}", "time / close": "{:.0%}",
       "mean stop (%)": "{:.2f}"}
display(pd.DataFrame(rows).T.style.format(FMT))
fig = go.Figure()
for (name, path), (_, (_, color)) in zip(paths.items(), EVENTS.items(), strict=False):
    fig.add_trace(go.Scatter(x=np.arange(HORIZON + 1), y=1e4 * path, mode="lines", name=name,
                             line={"color": color, "width": 2},
                             hovertemplate="+%{x} bars: %{y:+.1f} bps<extra>" + name + "</extra>"))
fig.add_hline(y=0, line={"color": "#898781", "width": 1})
plots._base(fig, title=f"<b>Fig 5</b> · {SYMBOL} — mean signed return after each raw trigger, "
                       "from the fill", height=420)
fig.update_layout(legend={"orientation": "v", "yanchor": "top", "y": 1.0, "xanchor": "left",
                          "x": 1.005}, margin={"r": 150})
fig.update_xaxes(title_text="bars after the fill")
fig.update_yaxes(title_text="bps")
fig.show()

## 6 · Is the construction stable? The anchor session's table under other knobs (Fig 6)

Back to the anchor session of Fig 2: the table gives its level table under other candle sizes and bin widths — how many levels appear, and the nearest support / resistance at the session's first close — and **Fig 6** draws the levels near the price under three candle sizes. A construction that produces a level every few ticks at one setting and none at the next is telling you the knob matters more than the tape; a stable pair of nearest levels across settings is what a "real" level looks like.

In [ ]:
grid = [(tf, bp) for tf in (15, 30, 60) for bp in (0.002, 0.005, 0.01)]
rows = {}
for tf, bp in grid:
    y = explain_levels(panel, SYMBOL, SESSION, **{**PRM, "tf_levels": tf, "bin_pct": bp})
    first = pd.Timestamp(str(y["session"]), tz=panel.tz).tz_convert("UTC")
    at = y["support"].loc[first:].index[0]
    rows[(f"{tf}m candles", f"bin {bp:.1%}")] = {
        "levels": len(y["levels"]), "confirmed swings": int(y["pivots"]["counted"].sum()),
        "support at open": y["support"].loc[at], "resistance at open": y["resistance"].loc[at],
        "touches (sup/res)": f"{y['sup_touches'].loc[at]:.0f} / {y['res_touches'].loc[at]:.0f}"}
table = pd.DataFrame(rows).T
display(table.style.format({"support at open": "{:.2f}", "resistance at open": "{:.2f}"}))

fig = make_subplots(rows=1, cols=3, shared_yaxes=True, horizontal_spacing=0.03,
                    subplot_titles=[f"{tf}-minute candles, bin {p['bin_pct']:.1%}"
                                    for tf in (15, 30, 60)])
day_bars = close[sessions.to_numpy() == day]
for k, tf in enumerate((15, 30, 60), 1):
    y = explain_levels(panel, SYMBOL, SESSION, **{**PRM, "tf_levels": tf})
    fig.add_trace(go.Scatter(x=day_bars.index, y=day_bars, mode="lines", name="close",
                             showlegend=k == 1, line={"color": "#0b0b0b", "width": 1}),
                  row=1, col=k)
    for _, r in y["levels"].iterrows():
        if day_bars.min() * 0.97 <= r["level"] <= day_bars.max() * 1.03:
            fig.add_shape(type="line", x0=day_bars.index[0], x1=day_bars.index[-1], y0=r["level"],
                          y1=r["level"], row=1, col=k,
                          line={"color": plots.SERIES[0], "width": 1 + 0.5 * min(r["touches"], 6)})
plots._base(fig, title=f"<b>Fig 6</b> · {SYMBOL} · {day}: the levels near the price under three "
                       "candle sizes", height=440)
fig.update_layout(showlegend=False, margin={"t": 90})
fig.update_annotations(yshift=6)
fig.update_xaxes(type="date", rangebreaks=plots._intraday_breaks(day_bars.index))
fig.show()

## Read it honestly

* Everything here is **one name, uncapped, in-sample, with no market removed**. It can show that the mechanics are wrong; it cannot show that the strategy is right — that is [`level_flow.ipynb`](./level_flow.ipynb) §5b and the diagnostics notebook.
* If the levels look like noise at the shipped `bin_pct` / `tf_levels`, §6 (Fig 6) is where to see whether any setting makes them look like levels — and if none does on a volatile name, the construction is the problem, not the parameters.
* A bounce marker sitting *below* its support line, or a break marker on the wrong side (Fig 3, Figs 4b/4c), is a bug: report it, do not tune around it.
* In Figs 4b/4c the stop must sit on the far side of the level from the fill, outside the shaded band; the target must be `tp_ratio` stop distances away on the near side.